### GPT-2 model breakdown - what do we need?

- Token Embedding  - GPT-2 use byte pair encoding BPE (https://huggingface.co/learn/llm-course/en/chapter6/5)

- Positional Encoding

- Transformer Block (Drcoder only)
    - Multi-head Attention
    - Layer Normalization
    - Feed Forward Neural Network (FFN)
    - Residual Connection

In [8]:
%pip install transformers

  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 1.1 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 1.4 MB/s  0:00:02 eta 0:00:010m
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
  Attempting uninstall: typer
    Found existing installation: typer 0.26.7
    Uninstalling typer-0.26.7:
      Successfully uninstalled typer-0.26.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [transformers] [transformers]ub]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset
from tqdm.notebook import tqdm


In [2]:
# 0 tokenize text
from transformers import GPT2Tokenizer

# Initialize the BPE tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [3]:
text = "Today is Sunny."
encoded_input = tokenizer.encode(text, return_tensors='pt')  # Returns a tensor
print(f"Encoded input: {encoded_input}")

Encoded input: tensor([[ 8888,   318, 32241,    13]])


In [4]:
import torch
import torch.nn as nn
import math

# Configuration for GPT-2 124M
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [5]:
# token embedding - > covert tokens to numeric values
class Embedding(nn.Module):
    def __init__(self, vocab_size, embed_size): # initialie the parent class of nn.Module
        super().__init__()
        # Initialize the embedding layer with the specified vocabulary size and embedding dimension
        self.embed = nn.Embedding(vocab_size, embed_size) # this is where learning happens
        # internally pytorch create a matrix of W ∈ R vocab_size×embed_size

    def forward(self, x): # this what happens when you call the module
        #  so when you write Embedding(x) -> internally pytorch calls Embedding.forward(x) with extra tracking for gradients
        # Forward pass: convert input token IDs to their corresponding embeddings
        return self.embed(x)


In [6]:
# Test Embedding
# Create an instance of the Embedding layer using the configuration values
embedding = Embedding(GPT_CONFIG_124M["vocab_size"], GPT_CONFIG_124M["emb_dim"])
# Generate random input token IDs with shape (batch_size, seq_length)
input_ids = torch.randint(0, GPT_CONFIG_124M["vocab_size"], (2, 10))
# Apply the embedding layer to the input token IDs
embed_output = embedding(input_ids)
# Print the shape of the output embeddings
print(f"Embedding output shape: {embed_output.shape}")
# Assert that the output shape matches the expected shape
# Expected shape: (batch_size, seq_length, embed_size)
assert embed_output.shape == (2, 10, GPT_CONFIG_124M["emb_dim"]), "Embedding shape mismatch"

Embedding output shape: torch.Size([2, 10, 768])


In [7]:
# now positional encoding 
#  I have explained positional encoding in transformer folder, if you need to refresh your memory please visiti there.
class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_seq_len = 512):
        super().__init__()
        #  intialize the tensor to hold the positional encodings
        pe = torch.zeros(max_seq_len, embed_size)
        
        # create a tensor for positions (0 to max_seq_length)
        position = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)

        # calculate  the division term for sin and cosine functions (create frequency scalling term),
        # creates dofferent waveengths for different dimentions
        div_term = torch.exp(torch.arange(0, embed_size, 2).float() * -(math.log(10000.0) / embed_size))

        # apply sin to even indices and cosine to add indicies
        pe[:, 0::2] = torch.sin(position * div_term) # sin for even indicies
        pe[:, 1::2] = torch.cos(position * div_term) # cos for odd indicies

        # register the token embediddings as a buffer (not a model prarameter)
        self.register_buffer('pe', pe.unsqueeze(0)) # Shape: (1, max_seq_length, embed_size)

    def forward(self, x):
        #  add the positional encoding to the input embeddings
        return x +  self.pe[:, :x.size(1)]


# why use register_buffer as instead of parameters?
ok so we need to understand what is parameter, nn.Paramater(), means its trainable via gradient descent -> this means optimizer does W <- W- lr * grad

however, in positioanl encoding  its a fixed mathematical fucntion which is sin/cos values. so these should not change.
buffer = non-trainable part of the model stats

In [8]:
# Test Positional Encoding
# Create an instance of the PositionalEncoding layer using the configuration values
pos_encoding = PositionalEncoding(GPT_CONFIG_124M["emb_dim"], GPT_CONFIG_124M["context_length"])
# Apply the positional encoding to the output of the embedding layer
pos_output = pos_encoding(embed_output)
# Print the shape of the output after adding positional encodings
print(f"Positional Encoding output shape: {pos_output.shape}")
# Assert that the output shape matches the expected shape
assert pos_output.shape == embed_output.shape, "Positional Encoding shape mismatch"

Positional Encoding output shape: torch.Size([2, 10, 768])


In [9]:
# lets implement multi-head attention mechanism now
# x → Q, K, V → attention scores → weighted sum → output
class MultiHeadAttention(nn.Module):
    # embed_size = size of the token vector(e.g. 768)
    # num_heads = how many attention heads (e.g. 12)
    def __init__(self, embed_size, num_heads, qkv_bias = False):
        super().__init__()
        #  split the representation  -  so each head sees only one part of the vector
        #  instead of one big attention we run (768 / 12) = 64 -> we run 12 small attention in parallel
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads


        # in this step we are learning 3 different transaformation
        # each of these is a matrix Wq, Wk, Wv ∈ (embed_size × embed_size)
        # these are learnable projections
        self.query = nn.Linear(embed_size, embed_size, bias= qkv_bias) # what I am looking for
        self.key = nn.Linear(embed_size, embed_size, bias= qkv_bias) # what I contain
        self.value = nn.Linear(embed_size, embed_size, bias= qkv_bias) # what I will give

        # output projection 
        # why we need this? after combining heads, we must mix information from all heads back togther
        self.out = nn.Linear(embed_size, embed_size)
    
    def forward(self, x, mask=None):
        batch_size = x.shape[0]

        q = self.query(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.key(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.value(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)


        attention = torch.matmul(q, k.transpose(-1,-2)) / math.sqrt(self.head_dim)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, float('-inf'))
        attention = torch.softmax(attention, dim=-1)
        
        out = torch.matmul(attention, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.embed_size)
        return self.out(out)



Instead of one big attention we run (768 / 12) = 64 -> we run 12 small attention in parallel
Each head sees ALL tokens, not a subset of the dataset or sequence.It only sees a different learned projection of the same tokens. Also, Each head is NOT given a different part of the sequence. Instead, every head sees ALL tokens in the sequence, but through a different learned lens.

In [10]:
# Test Multi-Head Attention
mha = MultiHeadAttention(GPT_CONFIG_124M["emb_dim"], GPT_CONFIG_124M["n_heads"])
mha_output = mha(pos_output)
print(f"Multi-Head Attention output shape: {mha_output.shape}")
assert mha_output.shape == pos_output.shape, "Multi-Head Attention shape mismatch"

Multi-Head Attention output shape: torch.Size([2, 10, 768])


Layer normalization plays a key role in stabilizing and imporving the training of GPT-2 model. It ensures that modle can handle large -scale data more effiecentlt y nirmalising the inout of each layer.The main different in GPT-2 model is that layer normalisation is applied before attention, and ffn layer this helps to prevent vanishing / exploading diagram and improve convergece during traing.


It normalize te hidden unit activations within the later, ensuring they ahve zerop mean and unit variance


Layer Normalization, normalise across the feature dimension




**what layer norm does?**
for each token vector (x = [x1, x2, ..., xN]), it make sure we hve mean = 0, and variance =1, then allowa the model to learn how to re-adjust it


We allow the model to learn:

scale -> how much each feature should matter

shift -> where the distribution should move


y=γ⋅x^+β

Where:

x^ = normalized input,

γ = scale, 

β = shift

In [11]:
#  implemment the layer normalization
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps = 1e-5):
        super().__init__()
        self.eps = eps # epsilon to avoid division by zero
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        # calculate mean and variance
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)

        # Normalise the input
        norm_x = (x - mean) / torch.sqrt(variance + self.eps)

        # Scale and shift
        return self.scale * norm_x + self.shift

forward pass os used to further process the attention outputs and capture more complex transformations.GPT-2 uses GELU(Guassian nError Linear Unit) activation function in its feed-forward layers.

each transformer block GPT-2 containes a feed-forward network that consist of two fully connected layers with non-linear activation functions (GELU) applied between them

even though attention mixes tokens together, this block, process each token independently

In [12]:
#  feed-forward network
class FeedForward(nn.Module):
    """
        emdbed_size = input size per token
        ff_hidden_size expanded internal size
    """
    def __init__(self, embed_size, ff_hidden_size):
        super().__init__()
        # First linear layer that transforms input from embedding size to hidden size
        self.fc1 = nn.Linear(embed_size, ff_hidden_size)
        # Second linear layer that transforms from hidden size back to embedding size
        self.fc2 = nn.Linear(ff_hidden_size, embed_size)
        # GELU activation function - It is a smooth nonlinear function - 
        # keeps positive values, smoothly suppresses negative ones
        self.gelu = nn.GELU()
    def forward(self, x):
        # Forward pass: apply the first linear layer, then GELU activation, and finally the second linear layer
        return self.fc2(self.gelu(self.fc1(x)))

In [13]:
# Test Feed-Forward Network
# Define the hidden size for the feed-forward network (4 times the embedding size)
ff_hidden_size = GPT_CONFIG_124M["emb_dim"] * 4
# Create an instance of the FeedForward network
ff = FeedForward(GPT_CONFIG_124M["emb_dim"], ff_hidden_size)
# Apply the FeedForward network to the output of the multi-head attention layer
ff_output = ff(mha_output)
# Print the shape of the output after applying the FeedForward network
print(f"Feed-Forward output shape: {ff_output.shape}")
# Assert that the output shape matches the expected shape
assert ff_output.shape == mha_output.shape, "Feed-Forward shape mismatch"

Feed-Forward output shape: torch.Size([2, 10, 768])


Residual connections are used around both the multi-head attention and feed-forward netwotk layers. they work by adding the input of a layer directly to its output and create a shortcut path for gradient during backpropagation. As a result, residual connections make it feasible to train much deeper networks, since models without them often have difficulty converging.


Residual connections solve a signal flow + optimization problem

y = x + F(x)

In [14]:
import torch
import torch.nn as nn

# GPT-2 Correct Transformer Block (Pre-LN)
class TransformerBlock(nn.Module):
    def __init__(self, embed_size, num_heads, ff_hidden_size, dropout=0.1, qkv_bias=False):
        super().__init__()

        # LayerNorm BEFORE attention (GPT-2 style)
        self.ln1 = nn.LayerNorm(embed_size)
        self.mha = MultiHeadAttention(embed_size, num_heads, qkv_bias)
        self.dropout1 = nn.Dropout(dropout)

        # LayerNorm BEFORE FFN (GPT-2 style)
        self.ln2 = nn.LayerNorm(embed_size)
        self.ff = FeedForward(embed_size, ff_hidden_size)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):

        # --- Attention block ---
        norm_x = self.ln1(x)
        attn_out = self.mha(norm_x, mask)
        x = x + self.dropout1(attn_out)

        # --- Feed-forward block ---
        norm_x = self.ln2(x)
        ff_out = self.ff(norm_x)
        x = x + self.dropout2(ff_out)

        return x

In [15]:
transformer = TransformerBlock(
    GPT_CONFIG_124M["emb_dim"],
    GPT_CONFIG_124M["n_heads"],
    ff_hidden_size
)

transformer_output = transformer(pos_output)

print(f"Transformer Block output shape: {transformer_output.shape}")

assert transformer_output.shape == pos_output.shape, "Transformer Block shape mismatch"

Transformer Block output shape: torch.Size([2, 10, 768])


In [16]:
class GPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding = Embedding(config["vocab_size"], config["emb_dim"])
        self.positional_encoding = PositionalEncoding(config["emb_dim"], config["context_length"])
        self.transformer_blocks = nn.ModuleList(
            [
                TransformerBlock(
                    config["emb_dim"],
                    config["n_heads"],
                    config["emb_dim"] * 4,
                    config["drop_rate"],
                    config["qkv_bias"],
                )
                for _ in range(config["n_layers"])
            ]
        )
        self.fc_out = nn.Linear(config["emb_dim"], config["vocab_size"])
        self.dropout = nn.Dropout(config["drop_rate"])
 
    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        for block in self.transformer_blocks:
            x = block(x, mask)
        return self.fc_out(x)

In [17]:
model = GPT2(GPT_CONFIG_124M)

input_ids = torch.randint(
    0,
    GPT_CONFIG_124M["vocab_size"],
    (2, 64)
)

output = model(input_ids)

print(f"GPT-2 Model output shape: {output.shape}")

assert output.shape == (2, 64, GPT_CONFIG_124M["vocab_size"]), "GPT-2 Model shape mismatch"

GPT-2 Model output shape: torch.Size([2, 64, 50257])


In [18]:

def train(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    num_epochs: int = 3,
    grad_clip: float = 1.0,
):
    model.train()
    criterion = nn.CrossEntropyLoss()
 
    for epoch in range(1, num_epochs + 1):
        epoch_loss = 0.0
 
        step_bar = tqdm(
            dataloader,
            desc=f"Epoch {epoch}/{num_epochs}",
            unit="batch",
        )
 
        for inputs, targets in step_bar:
            inputs  = inputs.to(device)
            targets = targets.to(device)
 
            mask   = make_causal_mask(inputs.size(1), device)
            logits = model(inputs, mask=mask)
 
            loss = criterion(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )
 
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
 
            epoch_loss += loss.item()
 
            # Update the inner bar with the current batch loss
            step_bar.set_postfix(loss=f"{loss.item():.4f}")
 
        avg_epoch_loss = epoch_loss / len(dataloader)
        print(f"\n{'─' * 50}")
        print(f"  Epoch     : {epoch}/{num_epochs}")
        print(f"  Avg Loss  : {avg_epoch_loss:.4f}")
        print(f"  Perplexity: {math.exp(avg_epoch_loss):.2f}")
        print(f"{'─' * 50}\n")
 

In [19]:
def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """
    Upper-triangular mask of shape (1, 1, T, T).
    Positions where mask == 0 will be filled with -inf in MHA.
    """
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)   # (1, 1, T, T)

In [20]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """
    Sliding-window dataset.
    Each sample is (input_ids[i : i+seq_len], input_ids[i+1 : i+seq_len+1]).
    The target is the input shifted by one token — standard LM objective.
    """
 
    def __init__(self, text: str, tokenizer, seq_len: int, max_samples: int = None):
        self.seq_len = seq_len
        print("Tokenizing text...")
        tokens = tokenizer.encode(text)
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        total = max(0, len(self.tokens) - self.seq_len)
        self.length = min(total, max_samples) if max_samples else total
        print(f"Total tokens: {len(self.tokens):,} | Samples: {self.length:,}\n")
 
    def __len__(self):
        return self.length
 
    def __getitem__(self, idx):
        chunk = self.tokens[idx : idx + self.seq_len + 1]
        return chunk[:-1], chunk[1:]  # (input, target)

In [21]:
 
def train(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    num_epochs: int = 3,
    grad_clip: float = 1.0,
):
    model.train()
    criterion = nn.CrossEntropyLoss()
 
    epoch_bar = tqdm(range(1, num_epochs + 1), desc="Training", unit="epoch")
 
    for epoch in epoch_bar:
        epoch_loss = 0.0
 
        step_bar = tqdm(
            dataloader,
            desc=f"Epoch {epoch}/{num_epochs}",
            leave=False,
            unit="batch",
        )
 
        for inputs, targets in step_bar:
            inputs  = inputs.to(device)
            targets = targets.to(device)
 
            mask   = make_causal_mask(inputs.size(1), device)
            logits = model(inputs, mask=mask)
 
            loss = criterion(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )
 
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
 
            epoch_loss += loss.item()
 
            # Update the inner bar with the current batch loss
            step_bar.set_postfix(loss=f"{loss.item():.4f}")
 
        avg_epoch_loss = epoch_loss / len(dataloader)
        epoch_bar.set_postfix(loss=f"{avg_epoch_loss:.4f}", perplexity=f"{math.exp(avg_epoch_loss):.2f}")
        print(f"\n{'─' * 50}")
        print(f"  Epoch     : {epoch}/{num_epochs}")
        print(f"  Avg Loss  : {avg_epoch_loss:.4f}")
        print(f"  Perplexity: {math.exp(avg_epoch_loss):.2f}")
        print(f"{'─' * 50}\n")

In [ ]:
#  generate output using GPT-2 model
def generate_text_simple(model, idx, max_new_tokens, context_size, device):
    """
    Generates new tokens autoregressively using greedy decoding.
    Args:
        model        (nn.Module)    : The GPT-2 model instance.
        idx          (torch.Tensor) : Prompt token IDs, shape (batch, T).
        max_new_tokens (int)        : Number of new tokens to generate.
        context_size (int)          : Maximum sequence length the model supports.
                                      Older tokens are dropped once exceeded.
        device       (torch.device) : Device to run inference on (cpu/cuda).

    Returns:
        torch.Tensor: Extended token IDs of shape (batch, T + max_new_tokens).

    Note:
        Uses greedy decoding — always picks the single most probable token.
        For more diverse output consider top-k or temperature sampling instead.
    """
    model.eval() # disables dropout so generation is deterministic
    idx = idx.to(device)
    for _ in range(max_new_tokens): # the generation loop
        idx_cond = idx[:, -context_size:]
        mask = make_causal_mask(idx_cond.size(1), device)
        with torch.no_grad():
            logits = model(idx_cond, mask=mask)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [31]:

# ── Hyperparameters ──────────────────────
SEQ_LEN    = 64       # tokens per training sample (keep small for quick testing)
BATCH_SIZE = 4
NUM_EPOCHS = 3
LR         = 3e-4     # AdamW default for GPT-style models
GRAD_CLIP  = 1.0
NUM_DOCS   = 5000  
MAX_SAMPLES = 10000    # cap sliding windows to keep training fast
                      # set to None to use all available samples

# Use a smaller config for fast local training; swap in GPT_CONFIG_124M
# once you have a GPU or want to run the full model.
SMALL_CONFIG = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [33]:
%pip install datasets


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [34]:
def load_openwebtext(num_docs: int = 5000) -> str:
    """
    Streams the first `num_docs` documents from OpenWebText
    (the open-source replication of GPT-2's training data).
    Uses streaming so the full 40GB dataset is never downloaded.
    """
    print(f"Streaming {num_docs} documents from OpenWebText...")
    
    owt = load_dataset("Skylion007/openwebtext", split="train", streaming=True, trust_remote_code=True)
    docs = list(owt.take(num_docs))
    text = "\n".join(doc["text"] for doc in docs)
    print(f"Loaded {num_docs} documents ({len(text):,} characters)\n")
    return text

In [35]:
# ── Tokenizer ────────────────────────────
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# ── Data ─────────────────────────────────
# Replace with any text file you like:
text       = load_openwebtext(num_docs=NUM_DOCS)
dataset    = TextDataset(text, tokenizer, seq_len=SEQ_LEN, max_samples=MAX_SAMPLES)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
print(f"Dataset: {len(dataset)} samples | Batches per epoch: {len(dataloader)}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Skylion007/openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Streaming 5000 documents from OpenWebText...


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loaded 5000 documents (24,434,929 characters)

Tokenizing text...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (5593290 > 1024). Running this sequence through the model will result in indexing errors


Total tokens: 5,593,290 | Samples: 10,000

Dataset: 10000 samples | Batches per epoch: 2500


In [36]:
# ── Model ────────────────────────────────
model     = GPT2(SMALL_CONFIG).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}\n")

Trainable parameters: 13,311,825



In [37]:
# ── Train ────────────────────────────────
train(
    model,
    dataloader,
    optimizer,
    device,
    num_epochs=NUM_EPOCHS,
    grad_clip=GRAD_CLIP,
)

# ── Save checkpoint ──────────────────────
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": SMALL_CONFIG,
    },
    "gpt2_checkpoint.pt",
)
print("Checkpoint saved to gpt2_checkpoint.pt")

Training:   0%|          | 0/3 [00:00<?, ?epoch/s]

Epoch 1/3:   0%|          | 0/2500 [00:00<?, ?batch/s]


──────────────────────────────────────────────────
  Epoch     : 1/3
  Avg Loss  : 3.4007
  Perplexity: 29.99
──────────────────────────────────────────────────



Epoch 2/3:   0%|          | 0/2500 [00:00<?, ?batch/s]


──────────────────────────────────────────────────
  Epoch     : 2/3
  Avg Loss  : 0.9812
  Perplexity: 2.67
──────────────────────────────────────────────────



Epoch 3/3:   0%|          | 0/2500 [00:00<?, ?batch/s]


──────────────────────────────────────────────────
  Epoch     : 3/3
  Avg Loss  : 0.4636
  Perplexity: 1.59
──────────────────────────────────────────────────

Checkpoint saved to gpt2_checkpoint.pt


In [60]:
# ── Quick generation test ─────────────────
prompt = "I have to study for"
encoded = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
output  = generate_text_simple(model, encoded, max_new_tokens=3,
                               context_size=SMALL_CONFIG["context_length"],
                               device=device)
print("\nGenerated text:")
print(tokenizer.decode(output.squeeze(0).tolist()))


Generated text:
I have to study for a musical imprint
